# Compare the old and trajectory-trained Pi0.5 transcoders

Upload this notebook to Colab, select **L4 or A100**, and run top to bottom. Both checkpoint paths are filled in. No live browser control is needed. The exact source is bundled here; extraction writes only to a separate comparison directory.

The default **full** matches the original experiment notebook's controls: ten tasks, two scene seeds, twelve states, two noise samples, two perturbation doses, twenty target circuits, twenty top examples, two parents per node, depth three, sixty nodes, and 2,000 random audit draws. Discovery uses episodes 0–4 and forty batches. Allow many hours. The optional **pilot** is only a smoke test. Both checkpoints are rerun with the same bundled code, teacher revision, seeds, and budgets, so the primary comparison is between these two fresh runs. Previously saved results may have used different settings and are not automatically interchangeable.

Fresh feature discovery is performed separately for each dictionary. Feature numbers do not correspond across models.

1. Check architecture and file hashes.
2. Compare native predictions with **all 18 MLPs replaced together**, using matched initial observations and noise.
3. Measure object/prompt selectivity, nominate features by the same rule, trace, and audit parents against random controls.
4. Read the final side-by-side table and saved `comparison.json`.

This is an **exploratory comparison**. The audit tests object-related content in circuits, not whether a feature causally controls reaching. Nomination and audit reuse scenes, and discovered top examples differ between models. A causal/generalization claim still requires separate held-out interventions. Existing per-model binomial intervals assume independent circuits, which shared tasks/features can violate.

Subprocesses isolate GPU memory between stages and stream their Python tracebacks into logs. Stage failures are recorded, never silently counted as successful circuits. Rerunning a stage recomputes it; rerun Controls for a fresh run ID.


In [ ]:
# @title Comparison controls
from pathlib import Path
import time

PROFILE = "full" # @param ["full", "pilot"]
DRIVE_ROOT = "/content/drive/MyDrive/groot-run-shared-programmer908" # @param {type:"string"}
OLD_CHECKPOINT = "transcoders/pi05_libero/allframes_80-10-10_epoch1_b8_exp16_latest_lambda1e-4/step_027233.pt"
NEW_CHECKPOINT = "outputs/trajectory-training/pi05-trajectory-transcoders-001/trained_transcoders.pt"
NEW_SHA256 = "ea582a91af113f20c232a545ebc11f47c5e2d6f115929ae7a3a96ccd0f74784e"
POLICY_PATH = "lerobot/pi05_libero_finetuned"
SUITE = "libero_spatial"
SEED = 1000
TASK_ID_LIST = [0,7,8,9] if PROFILE == "pilot" else list(range(10))
SEED_LIST = [1000] if PROFILE == "pilot" else [1000,1001]
STATES = 4 if PROFILE == "pilot" else 12
STATE_STRIDE = 4
NOISE_SAMPLES = 2
TARGET = PLACEBO_TARGET = ""
LIST_OBJECTS_ONLY = False
PERTURBATION, COLOR, DOSE = "blend", "1.0,0.2,0.1", "0.5,1.0"
MIN_TRACE_CONSISTENCY, MIN_TRACE_LAYER, MIN_SELECTIVITY = 0.25, 4, 2.0
RUN_CIRCUIT_TRACE = RUN_AUDIT_CALIBRATION = SAVE_RUNS_TO_DRIVE = True
DISCOVERY_EPISODES, DISCOVERY_MAX_BATCHES = "0,1,2,3,4", 40
TRACE_N_TARGETS = 4 if PROFILE == "pilot" else 20
TRACE_TOP_EXAMPLES = 5 if PROFILE == "pilot" else 20
TRACE_PARENTS_PER_NODE, TRACE_MAX_DEPTH = 2, 3
TRACE_MAX_NODES = 30 if PROFILE == "pilot" else 60
TRACE_SOURCE_POLICY = "previous-layer"
VALIDATION_RANDOM_DRAWS = 2000
MIN_GPU_MEMORY_GB = 20
FIDELITY_STARTS = "0" if PROFILE == "pilot" else "0,1,2"
RUN_ID = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
CHECKPOINTS = {"old": OLD_CHECKPOINT, "new": NEW_CHECKPOINT}
RESULTS = {}
print(f"{PROFILE}: {len(TASK_ID_LIST)} tasks x {len(SEED_LIST)} seeds per model; "
      f"up to {TRACE_N_TARGETS} circuits/model. Both models use identical budgets.")
print("This runs BOTH models from scratch. Full tracing can take many hours.")
print("Run ID:", RUN_ID)


In [ ]:
# @title Validate Runtime And Mount Drive

import os
import shutil
import subprocess
import time
from pathlib import Path

from google.colab import drive


def gpu_info():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        ).stdout.strip().splitlines()[0]
        name, mem = [part.strip() for part in out.split(",")]
        return name, int(mem) // 1024
    except Exception:
        return None, 0


gpu_name, gpu_mem_gb = gpu_info()
if gpu_name is None:
    raise RuntimeError("No GPU. Runtime > Change runtime type > L4.")
print(f"GPU: {gpu_name} (~{gpu_mem_gb} GiB)")
if gpu_mem_gb < int(MIN_GPU_MEMORY_GB):
    raise RuntimeError(
        f"{gpu_name} has ~{gpu_mem_gb} GiB; Pi0.5 needs >= {MIN_GPU_MEMORY_GB} GiB. Use an L4 or A100."
    )


def mount_drive_with_retry(mountpoint: str = "/content/drive", attempts: int = 3) -> None:
    """Mount Drive, retrying because `mount failed` is usually transient."""
    if Path(mountpoint, "MyDrive").exists():
        print(f"Drive already mounted at {mountpoint}", flush=True)
        return
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            drive.mount(mountpoint, force_remount=attempt > 1)
            print(f"Drive mounted at {mountpoint} (attempt {attempt})", flush=True)
            return
        except Exception as exc:
            last_error = exc
            print(f"Drive mount attempt {attempt}/{attempts} failed: {exc}", flush=True)
            if attempt < attempts:
                delay = 5 * attempt
                print(f"  retrying in {delay}s", flush=True)
                time.sleep(delay)
    raise RuntimeError(
        f"Could not mount Google Drive after {attempts} attempts (last error: {last_error}).\n"
        "Drive is needed here only to read the transcoder checkpoint.\n"
        "Most likely causes, in order:\n"
        "  1. Other Colab sessions hold Drive mounts. Runtime > Manage sessions, terminate "
        "the ones you are not using, then Runtime > Restart session and rerun.\n"
        "  2. The authorization popup was blocked. Allow popups and third-party cookies.\n"
        "  3. A transient Drive outage. Restart the runtime and retry in a few minutes."
    ) from last_error


mount_drive_with_retry()
DRIVE_ROOT = Path(DRIVE_ROOT)
print("Drive root:", DRIVE_ROOT, "exists:", DRIVE_ROOT.exists())

# Everything this notebook measures is written under DRIVE_ROOT/outputs, because
# Colab's disk is wiped when the runtime ends. If the folder is not actually
# there, copytree would happily create a fresh unshared directory and the run
# would look saved while being invisible to anyone else. Fail here instead.
DRIVE_OUTPUTS = DRIVE_ROOT / "outputs"
if SAVE_RUNS_TO_DRIVE:
    if not DRIVE_ROOT.exists():
        raise RuntimeError(
            f"{DRIVE_ROOT} does not exist under the mounted Drive.\n"
            "Check DRIVE_ROOT in Controls against the shared folder, and make sure the "
            "folder is added to *your* My Drive (Shared with me is not mounted).\n"
            "Set SAVE_RUNS_TO_DRIVE = False only if you accept losing the run."
        )
    DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
    probe = DRIVE_OUTPUTS / ".write_test"
    probe.write_text("ok")          # a read-only share fails here, not four hours in
    assert probe.read_text() == "ok"
    probe.unlink()
    free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1024**3
    print(f"Drive writable. Free on the mount: {free_gb:.1f} GB")
    print(f"Everything this notebook produces lands under {DRIVE_OUTPUTS}")
else:
    print("SAVE_RUNS_TO_DRIVE is off: this run will be lost when the runtime ends.")


In [ ]:
# @title Install / Repair Runtime (rerun safely after an interrupted install)
import os, subprocess, shlex
from collections import deque
from pathlib import Path

VENV = Path('/content/pi05-comparison-venv')
PYTHON = VENV / 'bin/python'
UV_BIN_DIR = Path('/content/uv-bin')
UV = str(UV_BIN_DIR / 'uv')
INSTALL_LOG = Path('/content/pi05-comparison-install.log')
FORK_COMMIT = 'dcddb970176382c0fcf4521b0c0e6fc15894dfe0'

def run(cmd, *, env=None):
    cmd = list(map(str, cmd))
    print('$', shlex.join(cmd), flush=True)
    tail = deque(maxlen=100)
    with INSTALL_LOG.open('a') as log:
        log.write('\n$ ' + shlex.join(cmd) + '\n')
        with subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            try:
                for line in proc.stdout:
                    print(line, end='', flush=True)
                    log.write(line); log.flush(); tail.append(line)
                status = proc.wait()
            except BaseException:
                proc.terminate()
                try: proc.wait(timeout=5)
                except subprocess.TimeoutExpired: proc.kill(); proc.wait()
                raise
    if status:
        raise RuntimeError(f'Command failed (exit {status}); log: {INSTALL_LOG}\n' + ''.join(tail))
    return subprocess.CompletedProcess(cmd, status)

os.environ['PATH'] = f'{VENV / "bin"}:{UV_BIN_DIR}:' + os.environ['PATH']
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '0'
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'Agg'

# A Python executable alone does not mean a previous install succeeded.
CHECK = r"""
import json, importlib.metadata as md, importlib.util
expected = {'lerobot':'0.4.4','torch':'2.10.0','torchvision':'0.25.0',
            'hf-libero':'0.1.4','numpy':'2.4.6','matplotlib':'3.10.0','av':'15.1.0'}
for name, value in expected.items():
    assert md.version(name).split('+')[0] == value, (name, md.version(name), value)
origin = json.loads(md.distribution('transformers').read_text('direct_url.json') or '{}')
assert origin.get('vcs_info',{}).get('commit_id') == 'dcddb970176382c0fcf4521b0c0e6fc15894dfe0'
from transformers.models.siglip import check
assert check.check_whether_transformers_replace_is_installed_correctly()
from lerobot.envs.configs import LiberoEnv
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
import torch, matplotlib, av, numpy
assert torch.cuda.is_available(), 'CUDA unavailable in comparison Python'
assert all(importlib.util.find_spec(n) for n in ('libero','egl_probe'))
print('Validated Pi0.5 fork, simulation imports, CUDA, and versions:', expected)
"""
ready = False
if PYTHON.exists():
    probe = subprocess.run([str(PYTHON), '-c', CHECK], capture_output=True, text=True)
    ready = probe.returncode == 0
    print(probe.stdout if ready else 'Incomplete/incompatible runtime; repairing.\n' + probe.stderr[-2500:])
if not ready:
    apt_env = {**os.environ, 'DEBIAN_FRONTEND':'noninteractive'}
    run(['apt-get','update','-qq'], env=apt_env)
    run(['apt-get','install','-y','-qq','build-essential','ca-certificates','cmake',
         'curl','ffmpeg','git','pkg-config','libegl1','libgl1','libglib2.0-0',
         'libglvnd0','libglx0','libopengl0','libosmesa6-dev','libsm6','libxext6','libxrender1'],env=apt_env)
    if not Path(UV).exists():
        UV_BIN_DIR.mkdir(parents=True,exist_ok=True)
        run(['curl','-LsSf','https://astral.sh/uv/install.sh','-o','/tmp/install-uv.sh'])
        run(['sh','/tmp/install-uv.sh'], env={**os.environ,'UV_INSTALL_DIR':str(UV_BIN_DIR)})
    if not PYTHON.exists():
        run([UV,'python','install','3.12'])
        run([UV,'venv',str(VENV),'--python','3.12'])
    # Match training: egl-probe requires pre-CMake-4 build tooling.
    run([UV,'pip','install','--python',PYTHON,'cmake==3.31.6','setuptools>=71,<81','wheel'])
    run([UV,'pip','install','--python',PYTHON,'--no-build-isolation-package','egl-probe','egl-probe==1.0.2'])
    override = Path('/content/pi05-comparison-transformers-overrides.txt')
    override.write_text(f'transformers @ git+https://github.com/huggingface/transformers.git@{FORK_COMMIT}\n')
    run([UV,'pip','install','--python',PYTHON,'--torch-backend','cu128',
         '--overrides',override,'--no-build-isolation-package','egl-probe','cmake==3.31.6',
         'lerobot[libero]==0.4.4','torch==2.10.0','torchvision==0.25.0',
         'hf-libero==0.1.4','hf-transfer','opencv-python','numpy==2.4.6',
         'matplotlib==3.10.0','av==15.1.0','scikit-learn','joblib','pillow'])
run([PYTHON,'-c',CHECK])
print('Runtime ready. All model stages run in this separate Python; no kernel restart needed.')


In [ ]:
# @title Restore the bundled source (no GitHub update required)
import base64, hashlib, json, os, subprocess, zlib
from pathlib import Path
SOURCE_SHA256 = 'd918514eaa4c8fd22752f046091f8bd16c66ff165672e4d16f7d9eb426192c68'
SOURCE_BASE_COMMIT = '79ccf32a10df551a0d33167d43e03cee85241f18'
payload = '...'
raw = zlib.decompress(base64.b64decode(payload))
assert hashlib.sha256(raw).hexdigest() == SOURCE_SHA256
LOCAL_REPO = Path("/content/pi05-comparison-source") / SOURCE_SHA256[:12]
for name, contents in json.loads(raw).items():
    target = LOCAL_REPO / name
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(contents)
COMPARISON_ROOT = LOCAL_REPO / "outputs/comparisons" / RUN_ID
COMPARISON_ROOT.mkdir(parents=True,exist_ok=True)
(COMPARISON_ROOT / "source_snapshot.json").write_bytes(raw)
(COMPARISON_ROOT / "source_manifest.json").write_text(json.dumps(dict(
    sha256=SOURCE_SHA256, base_commit=SOURCE_BASE_COMMIT, profile=PROFILE),indent=2))
os.environ["PYTHONPATH"] = os.pathsep.join([str(LOCAL_REPO/'src'),str(LOCAL_REPO/'scripts')])
def save_to_drive(path):
    import shutil
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(path, DRIVE_ROOT / path.relative_to(LOCAL_REPO), dirs_exist_ok=True)
print("Source:",LOCAL_REPO)


In [ ]:
# @title Check evaluation imports before running either model
preflight_code = """
import importlib, sys
from pi05_mi.lerobot_runtime import patch_transformers_causal_mask_compat, patch_pi05_checkpoint_key_compat
patch_transformers_causal_mask_compat()
patch_pi05_checkpoint_key_compat()
for name in ('probe_pi05_transcoder_counterfactual', 'compare_transcoder_fidelity',
             'collect_pi05_transcoder_features', 'trace_pi05_transcoder_circuit'):
    importlib.import_module(name)
    print('Import OK:', name, flush=True)
assert 'train_pi05_transcoders' not in sys.modules
print('Evaluation import preflight passed.', flush=True)
"""
run([PYTHON, '-u', '-c', preflight_code])


In [ ]:
# @title HF Token And LIBERO Assets

import os
from pathlib import Path

token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
if not token:
    secret_file = DRIVE_ROOT / "secrets/HF_TOKEN.txt"
    if secret_file.exists():
        token = secret_file.read_text().strip()
if not token:
    raise RuntimeError(
        "No Hugging Face token. Add HF_TOKEN as a Colab secret (key icon in the sidebar), "
        f"or place it at {DRIVE_ROOT / 'secrets/HF_TOKEN.txt'}. "
        f"{POLICY_PATH} is gated, so the download needs it."
    )
os.environ["HF_TOKEN"] = token
print("HF token loaded.")

# LIBERO ships without its scene assets; fetch them into the installed package.
assets_code = r"""
import shutil, site, os
from pathlib import Path
from huggingface_hub import snapshot_download

roots = [Path(p) for p in site.getsitepackages()]
user_site = site.getusersitepackages()
if user_site:
    roots.append(Path(user_site))
libero_root = next((r / "libero" / "libero" for r in roots if (r / "libero" / "libero").exists()), None)
if libero_root is None:
    raise RuntimeError("Installed LIBERO package not found")
assets_dir = libero_root / "assets"
required = assets_dir / "scenes" / "libero_tabletop_base_style.xml"
if required.exists():
    print("LIBERO assets already present:", required)
else:
    print("Downloading LIBERO assets (cached files are reused)...", flush=True)
    snap = Path(snapshot_download(repo_id="lerobot/libero-assets", repo_type="dataset",
                                  token=os.environ.get("HF_TOKEN") or None))
    assets_dir.mkdir(parents=True, exist_ok=True)
    for child in snap.iterdir():
        if child.name == ".gitattributes":
            continue
        target = assets_dir / child.name
        if child.is_dir():
            shutil.copytree(child, target, dirs_exist_ok=True)
        else:
            shutil.copy2(child, target)
    if not required.exists():
        raise FileNotFoundError(f"LIBERO asset install failed; missing {required}")
    print("LIBERO assets installed:", required)
"""
run([str(PYTHON), "-u", assets_code])

# Freeze the teacher to the exact revision recorded by the new training run.
teacher_result = LOCAL_REPO / 'teacher_path.txt'
run([PYTHON, '-u', LOCAL_REPO / 'scripts/download_pi05_comparison_teacher.py',
     '--result-path', teacher_result])
POLICY_PATH = teacher_result.read_text().strip()
print('Pinned teacher:', POLICY_PATH)
# Parse the actual checkpoint config before launching any evaluation tasks.
run([PYTHON, '-u', '-c',
     "from argparse import Namespace; from pi05_mi.lerobot_runtime import _load_train_config; "
     "import sys; cfg=_load_train_config(Namespace(policy_path=sys.argv[1],local_files_only=True)); "
     "print('Teacher config OK:', type(cfg.policy).__name__)", POLICY_PATH])


In [ ]:
# @title Verify and stage both checkpoints
import hashlib, shutil
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(16*1024*1024),b''): h.update(chunk)
    return h.hexdigest()
CHECKPOINT_HASHES, LOCAL_CHECKPOINTS = {}, {}
for arm,relative in CHECKPOINTS.items():
    source=DRIVE_ROOT/relative
    if not source.is_file(): raise FileNotFoundError(source)
    local=Path('/content/pi05-comparison-checkpoints')/arm/source.name
    local.parent.mkdir(parents=True,exist_ok=True)
    print('Hashing',arm,'checkpoint...',flush=True)
    expected=sha256(source)
    if arm=='new' and expected!=NEW_SHA256:
        raise ValueError('New checkpoint does not match the completed, downloaded training bundle')
    if not local.exists() or sha256(local)!=expected:
        partial=local.with_suffix('.partial');shutil.copyfile(source,partial)
        if sha256(partial)!=expected: raise IOError('Checkpoint copy hash mismatch')
        partial.replace(local)
    CHECKPOINT_HASHES[arm]=expected;LOCAL_CHECKPOINTS[arm]=str(local)
    print(arm,local,expected)
run([PYTHON,'-c',"from pi05_mi.transcoder_checkpoints import load_transcoders; import sys; "
     "m=load_transcoders(sys.argv[1],layers=[0]); print(type(next(iter(m.values()))).__name__)",
     LOCAL_CHECKPOINTS['old']])
run([PYTHON,'-c',"from pi05_mi.transcoder_checkpoints import load_transcoders; import sys; "
     "m=load_transcoders(sys.argv[1],layers=[0]); print(type(next(iter(m.values()))).__name__)",
     LOCAL_CHECKPOINTS['new']])


## Old transcoder

In [ ]:
# @title Select old checkpoint
ARM='old'
TRANSCODER_DRIVE_PATH=CHECKPOINTS[ARM]
TRANSCODER_CHECKPOINT=LOCAL_CHECKPOINTS[ARM]
RESULTS[ARM]={'errors':{}}
run_dirs=[]


In [ ]:
# @title Old: Fidelity
import traceback, shutil
try:
    fidelity_dir=COMPARISON_ROOT/ARM/'fidelity'
    run([PYTHON,'-u',LOCAL_REPO/'scripts/compare_transcoder_fidelity.py',
         '--checkpoint',TRANSCODER_CHECKPOINT,'--output-dir',fidelity_dir,
         '--tasks',','.join(map(str,TASK_ID_LIST)), '--starts',FIDELITY_STARTS,
         '--policy-path',POLICY_PATH,'--suite',SUITE,'--seed',str(SEED)])

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Fidelity']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title Old: Probe
import traceback, shutil
try:
    # @title Run Counterfactual Probe

    import json
    import os
    import subprocess
    import sys
    import time
    from pathlib import Path
    from IPython.display import display, HTML, Markdown, Image as IPyImage

    batch_dir = COMPARISON_ROOT / ARM / "probe"
    batch_dir.mkdir(parents=True, exist_ok=True)


    import shutil

    DRIVE_BATCH = (DRIVE_ROOT / batch_dir.relative_to(LOCAL_REPO)) if SAVE_RUNS_TO_DRIVE else None


    def _tree_stats(root: Path) -> tuple[int, int]:
        files = [p for p in root.rglob("*") if p.is_file()]
        return len(files), sum(p.stat().st_size for p in files)


    def save_to_drive(local_dir: Path, *, required: bool = True) -> Path | None:
        """Mirror one directory under DRIVE_ROOT/outputs/, preserving its path.

        Colab's disk does not survive the runtime, and neither does a disconnect
        four hours into a twenty-run loop. Every artefact the paper's tables are
        filled from (the delta store, decision_metrics.json, validation.json,
        provenance.json, the traced graph, the audit calibration) lives in these
        directories, so each is copied the moment it is complete rather than at the
        end of the notebook.

        The copy is verified. A Drive FUSE mount that has run out of quota returns
        success on write and leaves a short file, so comparing the byte count is
        the only way to know the data is really there.
        """
        if not SAVE_RUNS_TO_DRIVE:
            return None
        local_dir = Path(local_dir)
        if not local_dir.exists():
            return None
        rel = local_dir.relative_to(LOCAL_REPO)
        target = DRIVE_ROOT / rel
        src_files, src_bytes = _tree_stats(local_dir)
        try:
            shutil.copytree(local_dir, target, dirs_exist_ok=True)
            dst_files, dst_bytes = _tree_stats(target)
            short = src_bytes - dst_bytes
            if dst_files < src_files or short > 0:
                raise IOError(
                    f"copy is short by {src_files - dst_files} files / {short / 1024**2:.1f} MB "
                    f"(Drive quota or a dropped mount)"
                )
        except Exception as exc:
            message = f"!! FAILED to save {rel} to Drive: {exc}"
            if required:
                raise RuntimeError(
                    message + f"\n   The data is still on local disk at {local_dir}. "
                    "Free space or remount, then re-run this cell's save."
                ) from exc
            print(message, flush=True)
            return None
        print(f"saved {rel} -> {target}  ({dst_files} files, {dst_bytes / 1024**2:.0f} MB)", flush=True)
        return target


    def _log_tail(path: Path, lines: int = 20) -> list[str]:
        """The end of a failed run's log, carried in the manifest.

        A return code says a run failed and nothing about why. The log directory is
        saved too, but the manifest is the file someone reads first.
        """
        try:
            return [line[-300:] for line in
                    path.read_text(errors="replace").splitlines()[-lines:]]
        except OSError:
            return []


    def write_manifest(**extra) -> None:
        """An index of the batch, written into Drive beside the runs.

        Further analysis should be able to start from the Drive folder alone,
        without the notebook or the Colab session that produced it.
        """
        if not SAVE_RUNS_TO_DRIVE:
            return
        manifest = {
            "batch": batch_dir.name,
            "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "drive_root": str(DRIVE_ROOT),
            "config": {
                "suite": SUITE, "task_ids": TASK_ID_LIST, "seeds": SEED_LIST,
                "states": STATES, "state_stride": STATE_STRIDE, "noise_samples": NOISE_SAMPLES,
                "perturbation": PERTURBATION, "color": COLOR, "dose": DOSE,
                "policy_path": POLICY_PATH, "transcoder_checkpoint": TRANSCODER_DRIVE_PATH,
                "trace_n_targets": TRACE_N_TARGETS, "trace_source_policy": TRACE_SOURCE_POLICY,
            },
            "runs": [],
            "failures": [
                {"task_id": t, "seed": s, "returncode": rc,
                 "dir": f"task{t:02d}_seed{s}",
                 "log_tail": _log_tail(batch_dir / f"task{t:02d}_seed{s}" / "probe.log")}
                for t, s, rc in failures
            ],
            **extra,
            "reading": {
                "per task": "task*/decision_metrics.json -> h1, h2",
                "pooled over tasks": "aggregate/aggregate.json -> summary",
                "delta store": "task*/latents/ (signed per-feature deltas, npz blocks)",
                "H3 rate": "h3/h3_rate.json -> summary",
                "audit sensitivity": "h3/calibration_*/audit_calibration.json -> summary",
            },
        }
        for run_dir in run_dirs:
            entry = {"dir": run_dir.name}
            for name, key in (("decision_metrics.json", "verdicts"),
                              ("provenance.json", "provenance"),
                              ("nominated_targets.json", "nominated")):
                path = run_dir / name
                if not path.exists():
                    continue
                data = json.loads(path.read_text())
                if key == "verdicts":
                    entry["h1"] = data["h1"]["verdict"]
                    entry["h2"] = data["h2"]["verdict"]
                    entry["sel_adj_l2"] = data["h1"]["pooled"].get("sel_adj_l2")
                elif key == "provenance":
                    entry["commit"] = data.get("git", {}).get("commit")
                    entry["dirty"] = data.get("git", {}).get("dirty")
                else:
                    entry["nominated"] = [r["feature_key"] for r in data.get("nominated") or []]
            entry["files"], entry["bytes"] = _tree_stats(run_dir)
            manifest["runs"].append(entry)
        (batch_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
        target = DRIVE_BATCH / "manifest.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(manifest, indent=2))
        print(f"manifest -> {target}", flush=True)

    def build_cmd(task_id, seed, run_dir):
        cmd = [
            str(PYTHON), "-u", "scripts/probe_pi05_transcoder_counterfactual.py",
            "--policy-path", POLICY_PATH,
            "--output-dir", str(run_dir),
            "--suite", SUITE,
            "--task-id", str(task_id),
            "--seed", str(seed),
            "--device", "cuda",
            "--policy-dtype", "bfloat16",
        ]
        if LIST_OBJECTS_ONLY:
            cmd.append("--list-objects")
            return cmd
        cmd += [
            "--checkpoint", TRANSCODER_CHECKPOINT,
            "--perturbation", PERTURBATION,
            "--color", COLOR,
            "--dose", DOSE,
            "--states", str(STATES),
            "--state-stride", str(STATE_STRIDE),
            "--noise-samples", str(NOISE_SAMPLES),
            "--noise-seed", str(seed),
        ]
        # Omitted flags let each task pick its own object of interest and sibling.
        if TARGET.strip():
            cmd += ["--target", TARGET.strip()]
        if PLACEBO_TARGET.strip():
            cmd += ["--placebo-target", PLACEBO_TARGET.strip()]
        return cmd


    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    # Colab's inline matplotlib backend is only valid inside the kernel process;
    # inheriting it breaks LIBERO's import. The script guards this too.
    env["MPLBACKEND"] = "Agg"
    env["MUJOCO_GL"] = "egl"
    env["PYOPENGL_PLATFORM"] = "egl"
    env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
        os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
    )


    def emit(chunk: bytes) -> None:
        """Write through to the cell output.

        Colab's sys.stdout is an ipykernel OutStream with no .buffer, so decode
        rather than assuming a binary stream exists.
        """
        stream = getattr(sys.stdout, "buffer", None)
        if stream is not None:
            stream.write(chunk)
        else:
            sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()


    def run_probe(task_id, seed):
        run_dir = batch_dir / f"task{task_id:02d}_seed{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = build_cmd(task_id, seed, run_dir)
        log_path = run_dir / "probe.log"
        print(f"\n{'=' * 78}\n== task {task_id}, seed {seed}\n$ {' '.join(cmd)}\n", flush=True)
        with log_path.open("wb") as log_handle:
            process = subprocess.Popen(cmd, cwd=LOCAL_REPO, env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
            try:
                while True:
                    chunk = process.stdout.read(4096)
                    if not chunk:
                        break
                    log_handle.write(chunk); log_handle.flush()
                    emit(chunk)
                rc = process.wait()
            except BaseException:
                # Never leave the probe running headless if the cell is interrupted.
                process.kill(); process.wait(); raise
        return rc, run_dir, log_path


    # One task failing must not discard the tasks that already ran: this is an
    # hour-long loop and a mid-run abort would cost all of it.
    started = time.time()
    run_dirs, failures = [], []
    plan = [(t, s) for t in TASK_ID_LIST for s in SEED_LIST]
    for index, (task_id, seed) in enumerate(plan, start=1):
        print(f"\n[{index}/{len(plan)}]  elapsed {(time.time() - started) / 60:.1f} min", flush=True)
        rc, run_dir, log_path = run_probe(task_id, seed)
        # Saved either way, and immediately rather than at the end of the loop. A
        # disconnect twelve runs in would otherwise take all twelve, and a failed
        # run's log is the only record of why it failed: on the first full run
        # task 5 failed and its log died with the runtime.
        save_to_drive(run_dir, required=False)
        if rc == 0:
            run_dirs.append(run_dir)
        else:
            failures.append((task_id, seed, rc))
            print(f"\n!! task {task_id} seed {seed} exited {rc}; continuing. Tail of {log_path}:", flush=True)
            for line in log_path.read_text(errors="replace").splitlines()[-12:]:
                print("   ", line[-300:], flush=True)

    elapsed = time.time() - started
    print(f"\n{'=' * 78}")
    print(f"{len(run_dirs)} of {len(plan)} runs completed in {elapsed / 60:.1f} min")
    if failures:
        print("failed:", failures)
    if not run_dirs:
        raise SystemExit("No task produced a run; nothing to aggregate.")
    out_dir = run_dirs[0]  # the trace stage below works from a single task's run

    if LIST_OBJECTS_ONLY:
        display(Markdown(
            "### Objects listed\n\nSet `LIST_OBJECTS_ONLY = False` to run the measurement. "
            "Override `TARGET` / `PLACEBO_TARGET` only if you do not want the task's own objects."
        ))
    else:
        display(Markdown("### Per-task verdicts"))
        rows = []
        for run_dir in run_dirs:
            try:
                d = json.loads((run_dir / "decision_metrics.json").read_text())
            except FileNotFoundError:
                continue
            h1, h2 = d["h1"], d["h2"]
            rows.append(
                f"<tr><td>{run_dir.name}</td>"
                f"<td>{len(h1['cells'])}</td>"
                f"<td>{h1['pooled']['sel_raw']:.2f}</td>"
                f"<td>{(h1['pooled']['sel_adj_l2'] or float('nan')):.2f}</td>"
                f"<td>{h1['verdict'][:28]}</td>"
                f"<td>{(h2.get('referent_check') or {}).get('referent_moved')}</td>"
                f"<td>{h2['verdict'][:34]}</td></tr>"
            )
        display(HTML(
            "<table><tr><th>run</th><th>cells</th><th>Sel raw</th><th>Sel adj</th>"
            "<th>H1</th><th>referent moved</th><th>H2</th></tr>" + "".join(rows) + "</table>"
        ))
        print("Single-task verdicts are descriptive. The pooled verdict is in the next cell.")

        shapes_path = out_dir / "observation_shapes.json"
        if shapes_path.exists():
            shapes = json.loads(shapes_path.read_text())
            display(Markdown("### Observation Shapes Sent To The Policy"))
            display(HTML(
                "<table><tr><th>key</th><th>shape</th><th>dtype</th></tr>"
                + "".join(f"<tr><td>{k}</td><td>{v['shape']}</td><td>{v['dtype']}</td></tr>"
                          for k, v in sorted(shapes.get("tensor_shapes", {}).items()))
                + f"<tr><td>noise</td><td>{shapes.get('noise_shape')}</td>"
                  f"<td>{shapes.get('noise_dtype')}</td></tr></table>"
            ))
            print("task:", repr(shapes.get("task")))

        decision_path = out_dir / "decision_metrics.json"
        if decision_path.exists():
            decision = json.loads(decision_path.read_text())
            h1, h2 = decision["h1"], decision["h2"]

            def _f(v, spec=".4g"):
                return "n/a" if v is None else format(float(v), spec)

            pooled, spread, ci = h1["pooled"], h1["spread"], h1["bootstrap_ci_95"]
            display(Markdown(f"### H1 -- object selectivity (task prompt, {len(h1['cells'])} cells)"))
            display(HTML(
                "<table><tr><th>quantity</th><th>target</th><th>placebo</th><th>ratio</th></tr>"
                f"<tr><td>D, layer response (latent L2)</td><td>{_f(pooled['D_target'])}</td><td>{_f(pooled['D_placebo'])}</td><td>{_f(pooled['sel_raw'])}</td></tr>"
                f"<tr><td>S, changed-pixel fraction</td><td>{_f(pooled['S_px_target'])}</td><td>{_f(pooled['S_px_placebo'])}</td><td>{_f(pooled['S_px_ratio'])}</td></tr>"
                f"<tr><td>S, image-difference norm</td><td>{_f(pooled['S_l2_target'])}</td><td>{_f(pooled['S_l2_placebo'])}</td><td>{_f(pooled['S_l2_ratio'])}</td></tr>"
                f"<tr><td>action relative L2</td><td>{_f(pooled['A_target'])}</td><td>{_f(pooled['A_placebo'])}</td><td>{_f(pooled['action_sel'])}</td></tr>"
                "</table>"
            ))
            display(HTML(
                "<table><tr><th>selectivity</th><th>pooled</th><th>cells mean ± sd</th><th>min .. max</th><th>95% CI</th></tr>"
                + "".join(
                    f"<tr><td>{label}</td><td>{_f(pooled[k])}</td><td>{_f(spread[k]['mean'])} ± {_f(spread[k]['sd'])}</td>"
                    f"<td>{_f(spread[k]['min'])} .. {_f(spread[k]['max'])}</td><td>[{_f(ci[k]['low'])}, {_f(ci[k]['high'])}]</td></tr>"
                    for k, label in (("sel_raw", "raw"), ("sel_adj_px", "adjusted, |P|"), ("sel_adj_l2", "adjusted, ||dI||"))
                )
                + "</table>"
            ))
            nf = h1["null_floor"]
            print(f"null floor D: mean={_f(nf['mean'])} max={_f(nf['max'])} -> {'near zero' if h1['null_floor_near_zero'] else 'NOT near zero'}")
            pc = h1.get("positive_control", {})
            if pc.get("n"):
                print(f"positive control (prompt swap) D={_f(pc['D'])} (n={pc['n']}): target is {_f(pc['target_over_positive'])} of it, "
                      f"placebo {_f(pc['placebo_over_positive'])}")
            cs = h1.get("cluster_spread", {})
            if cs.get("n"):
                sp = cs["sel_adj_l2"]
                print(f"clusters {cs['unit']}: n={cs['n']}  Sel adj ||dI|| min={_f(sp['min'])} max={_f(sp['max'])}")
            rb = h1.get("robustness", {})
            if rb:
                print(f"robustness: Sel rel={_f(rb['sel_rel'])}  per-layer geomean={_f(rb['sel_layer_geomean'])} "
                      f"[{_f(rb['sel_layer_min'])} .. {_f(rb['sel_layer_max'])}], {rb['layers_above_one']}/{rb['layers']} layers above 1; "
                      f"direction agrees: {rb['agree_in_direction']}")
            for kind, dr in h1["dose_response"].items():
                print(f"dose response [{kind}]: elasticity={_f(dr['elasticity_mean'])} monotone={dr['monotone_in_dose']}  "
                      + "  ".join(f"d={r['dose']:g}: D={_f(r['D'])}" for r in dr["rows"]))
            display(Markdown(f"**H1 verdict: {h1['verdict']}**  \n<small>rule: {h1['decision_rule']}</small>"))

            g = h2["grounding"]
            display(Markdown("### H2 -- referent versus position (prompt swap over identical pixels)"))
            display(HTML(
                "<table><tr><th>prompt</th><th>D target</th><th>D placebo</th><th>selectivity</th><th>cells</th></tr>"
                + "".join(
                    f"<tr><td>{p}</td><td>{_f(r['D_target'])}</td><td>{_f(r['D_placebo'])}</td><td>{_f(r['sel_raw'])}</td><td>{r['n_cells']}</td></tr>"
                    for p, r in h2["per_prompt"].items()
                )
                + "</table>"
            ))
            print(f"prompt grounding g: mean={_f(g['mean'])} min={_f(g['min'])} max={_f(g['max'])} (n={g['n']}, threshold {h2['grounding_threshold']}); "
                  f"perturbation action effect {_f(h2['perturbation_action_effect'])}")
            rc = h2.get("referent_check", {})
            if rc:
                print(f"referent check (action anchor A_target/A_placebo): task={_f(rc.get('behavioural_anchor_task'))} "
                      f"alt={_f(rc.get('behavioural_anchor_alt'))} -> referent moved: {rc.get('referent_moved')}")
            display(Markdown(f"**H2 verdict: {h2['verdict']}**  \n<small>rule: {h2['decision_rule']}</small>"))

        provenance_path = out_dir / "provenance.json"
        if provenance_path.exists():
            pv = json.loads(provenance_path.read_text())
            ck = (pv.get("transcoder_checkpoint") or {}).get("digest", "")
            print(f"provenance: commit {pv['git'].get('commit')} dirty={pv['git'].get('dirty')} "
                  f"lerobot={pv['packages'].get('lerobot')} torch={pv['packages'].get('torch')} "
                  f"device={pv['device'].get('gpu_name', pv['device'].get('device'))} checkpoint sha256={ck[:16]}")

        summary_path = out_dir / "counterfactual_summary.json"
        if summary_path.exists():
            summary = json.loads(summary_path.read_text())
            display(Markdown("### Counterfactual Activation Response (every measurement)"))
            display(HTML(
                "<table><tr><th>state</th><th>noise</th><th>prompt</th><th>kind</th><th>target</th><th>dose</th>"
                "<th>changed pixels</th><th>latent L2</th><th>action rel L2</th></tr>"
                + "".join(
                    "<tr><td>{s}</td><td>{n}</td><td>{p}</td><td>{k}</td><td>{t}</td><td>{d}</td><td>{px:.5f}</td>"
                    "<td>{lat}</td><td>{act:.5g}</td></tr>".format(
                        s=r["state_index"], n=r.get("noise_index", 0), p=r.get("prompt", "task"),
                        k=r["kind"], t=r["target"], d=r["dose"],
                        px=r["pixel"]["changed_pixel_fraction"],
                        lat=("n/a" if r["latent"].get("l2_delta_mean") is None
                             else "{:.5g}".format(r["latent"]["l2_delta_mean"])),
                        act=r["action_relative_l2"])
                    for r in summary.get("measurements", []))
                + "</table>"
            ))
            print("Read every row against the 'null' rows: that is the nondeterminism floor.")

        # One state's frames for the eye: every camera's baseline, the target and
        # placebo recolours at the top dose, and their difference images.
        pngs = sorted((out_dir / "images").glob("state0_*.png"))
        keep = [p for p in pngs if "baseline" in p.name or "dose1_" in p.name]
        for png in keep[:16]:
            display(Markdown(f"**{png.name}**"))
            display(IPyImage(filename=str(png)))

        # Rank features by selectivity: responds to the task object, not to the
        # visually matched control. Run on EVERY task, not just the first: the H3
        # rate is supposed to be pooled across tasks, and nominating from one run
        # makes it a rate over one scene no matter how the trace stage samples.
        nominated_counts = {}
        for index, run_dir in enumerate(run_dirs, start=1):
            report_cmd = [str(PYTHON), "-u", "scripts/report_pi05_counterfactual_features.py",
                          str(run_dir), "--top", "30",
                          "--min-consistency", str(MIN_TRACE_CONSISTENCY),
                          "--min-trace-layer", str(MIN_TRACE_LAYER),
                          "--min-selectivity", str(MIN_SELECTIVITY)]
            report = subprocess.run(report_cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
            path = run_dir / "nominated_targets.json"
            n = len(json.loads(path.read_text()).get("nominated") or []) if path.exists() else 0
            nominated_counts[run_dir.name] = n
            if index == 1:
                print("\n$", " ".join(report_cmd), flush=True)
                print(report.stdout)          # the full ranking, once, for the eye
            if report.returncode != 0:
                print(f"  !! report on {run_dir.name} exited {report.returncode}")
                print(report.stderr[-800:])
        total_nominees = sum(nominated_counts.values())
        print(f"\nnominees per run: " +
              "  ".join(f"{k.replace('task', 't')}:{v}" for k, v in nominated_counts.items()))
        print(f"{total_nominees} nominees across {len(run_dirs)} runs, "
              f"{sum(1 for v in nominated_counts.values() if v)} runs contributing.")
        if sum(1 for v in nominated_counts.values() if v) < 2:
            print("  WARNING: fewer than two runs nominated anything, so the H3 rate "
                  "cannot be a rate over tasks. Relax MIN_SELECTIVITY or MIN_TRACE_LAYER.")

    # Every run was copied as it finished; this re-copies whatever the report stage
    # added afterwards (candidate_features.csv, nominated_targets.json) and writes
    # the batch index.
    for run_dir in run_dirs:
        save_to_drive(run_dir)
    write_manifest()
    print("\nArtifacts:", batch_dir)
    if SAVE_RUNS_TO_DRIVE:
        print("On Drive:", DRIVE_BATCH)

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Probe']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title Old: Aggregate
import traceback, shutil
try:
    if not run_dirs: raise RuntimeError('Probe produced no successful runs')
    # @title Aggregate Across Tasks

    import json, os, subprocess, sys
    from pathlib import Path
    from IPython.display import display, HTML, Markdown

    agg_dir = batch_dir / "aggregate"
    cmd = [str(PYTHON), "-u", "scripts/aggregate_counterfactual_runs.py", *[str(d) for d in run_dirs],
           "--output-dir", str(agg_dir)]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr[-3000:]); raise SystemExit(f"aggregation exited {proc.returncode}")

    report = json.loads((agg_dir / "aggregate.json").read_text())
    s = report["summary"]

    def _f(v, spec=".2f"):
        return "n/a" if v is None else format(float(v), spec)

    display(Markdown(f"### H1 pooled over {s['tasks']} tasks ({s['cells_total']} cells)"))
    p, ci = s["pooled"], s["bootstrap_ci_95"]
    display(HTML(
        "<table><tr><th>quantity</th><th>pooled</th><th>task-cluster 95% CI</th></tr>"
        + "".join(
            f"<tr><td>{label}</td><td>{_f(p.get(k))}</td>"
            f"<td>[{_f(ci[k].get('low'))}, {_f(ci[k].get('high'))}]</td></tr>"
            for k, label in (("sel_raw", "raw selectivity"),
                             ("sel_adj_l2", "adjusted, image norm (decides)"),
                             ("sel_adj_px", "adjusted, pixel count")))
        + "</table>"
    ))
    print(f"tasks with adjusted selectivity above 1: {s['tasks_above_one']} of {s['tasks']}")

    display(Markdown("**Per task** (the spread here is what the pilot could not see)"))
    display(HTML(
        "<table><tr><th>task</th><th>cells</th><th>Sel raw</th><th>Sel adj</th><th>action sel</th></tr>"
        + "".join(
            f"<tr><td>{r['task_id']}</td><td>{r['n_cells']}</td><td>{_f(r['sel_raw'])}</td>"
            f"<td>{_f(r['sel_adj_l2'])}</td><td>{_f(r['action_sel'])}</td></tr>"
            for r in report["per_task"])
        + "</table>"
    ))
    display(Markdown(f"**H1 verdict: {s['verdict']}**  \n<small>rule: {s['decision_rule']}</small>"))

    display(Markdown("### H2, gated per task"))
    display(HTML(
        "<table><tr><th>task</th><th>g</th><th>anchor task</th><th>anchor alt</th>"
        "<th>referent moved</th><th>Sel task</th><th>Sel alt</th></tr>"
        + "".join(
            f"<tr><td>{r['task_id']}</td><td>{_f(r['grounding'],'.3f')}</td><td>{_f(r['anchor_task'])}</td>"
            f"<td>{_f(r['anchor_alt'])}</td><td><b>{r['referent_moved']}</b></td>"
            f"<td>{_f(r['sel_task'])}</td><td>{_f(r['sel_alt'])}</td></tr>"
            for r in report["h2_per_task"])
        + "</table>"
    ))
    display(Markdown(f"**H2 verdict: {s['h2_verdict']}**"))

    save_to_drive(agg_dir)
    print("\nPaste the two verdict lines and the per-task tables into the write-up.")

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Aggregate']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title Old: Trace
import traceback, shutil
try:
    if not run_dirs: raise RuntimeError('Probe produced no successful runs')
    # @title Trace And Audit Several Targets

    import csv, json, math, os, subprocess, sys, time
    from pathlib import Path
    from IPython.display import display, HTML, Markdown

    if not RUN_CIRCUIT_TRACE:
        print("RUN_CIRCUIT_TRACE is off; skipping.")
    else:
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"; env["MPLBACKEND"] = "Agg"
        env["MUJOCO_GL"] = "egl"; env["PYOPENGL_PLATFORM"] = "egl"
        env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
            os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

        def stream(label, cmd, log_path):
            print(f"\n== {label} ==\n$ {' '.join(map(str, cmd))}", flush=True)
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open("wb") as handle:
                proc = subprocess.Popen([str(c) for c in cmd], cwd=LOCAL_REPO, env=env,
                                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
                try:
                    while True:
                        chunk = proc.stdout.read(4096)
                        if not chunk:
                            break
                        handle.write(chunk); handle.flush()
                        out = getattr(sys.stdout, "buffer", None)
                        (out.write(chunk) if out is not None else
                         sys.stdout.write(chunk.decode("utf-8", "replace")))
                        sys.stdout.flush()
                    return proc.wait()
                except BaseException:
                    proc.kill(); proc.wait(); raise

        # --- Pool nominees across every task -------------------------------------
        # A rate computed from one task's nominees is a rate over that scene. Each
        # nominee keeps the run it came from, because its audit has to read the
        # delta store that measured it.
        by_run, seen = {}, set()
        for run_dir in run_dirs:
            path = run_dir / "nominated_targets.json"
            if not path.exists():
                continue
            rows = []
            for row in json.loads(path.read_text()).get("nominated") or []:
                key = row["feature_key"]
                if key in seen:      # the same feature nominated by two tasks is
                    continue         # one circuit, and would be traced once anyway
                seen.add(key)
                rows.append({"run_dir": run_dir, **row})
            if rows:
                rows.sort(key=lambda r: -float(r["target_z"]))
                by_run[run_dir] = rows
        if not by_run:
            raise RuntimeError(
                "No task nominated a target under the rule. Widen the probe (STATES, SEEDS, DOSE) "
                "or relax MIN_TRACE_CONSISTENCY / MIN_TRACE_LAYER / MIN_SELECTIVITY deliberately."
            )

        # Round-robin across tasks, strongest nominee of each first. Sorting the
        # pooled nominees by z instead lets the single most selective task take
        # every slot, which is what happened on the first run: all five targets
        # came from one scene and the "rate over the method" was a rate over that
        # scene. Round-robin makes single-task monopoly impossible whenever more
        # than one task nominated anything.
        targets, depth = [], 0
        while len(targets) < TRACE_N_TARGETS:
            layer = [rows[depth] for rows in by_run.values() if len(rows) > depth]
            if not layer:
                break
            layer.sort(key=lambda r: -float(r["target_z"]))
            targets.extend(layer[: TRACE_N_TARGETS - len(targets)])
            depth += 1
        pool = [row for rows in by_run.values() for row in rows]
        tasks_represented = len({str(r["run_dir"]) for r in targets})
        if tasks_represented < 2:
            print("  WARNING: every target comes from one task, so the rate below is a rate "
                  "over that scene and not over the method.")
        bound = 1 - 0.05 ** (1 / len(targets))
        print(f"{len(pool)} distinct nominees from {len(by_run)} of {len(run_dirs)} runs; "
              f"tracing {len(targets)} across {tasks_represented} task(s).")
        print(f"A null result over {len(targets)} bounds the corroboration rate below {bound:.1%}.\n")
        for row in targets:
            print(f"  {row['feature_key']:<22} z={float(row['target_z']):+.2f}  "
                  f"sel={float(row['selectivity']):.1f}x  from {row['run_dir'].name}")

        batch_out = batch_dir / "h3"
        batch_out.mkdir(parents=True, exist_ok=True)

        # --- Discovery once, shared by every trace -------------------------------
        stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
        feature_dir = LOCAL_REPO / "outputs/features/pi05_libero" / f"comparison-{RUN_ID}-{ARM}"
        rc = stream("Feature discovery", [
            PYTHON, "-u", "scripts/collect_pi05_transcoder_features.py",
            "--policy-path", POLICY_PATH, "--seed", str(SEED),
            "--checkpoint", TRANSCODER_CHECKPOINT, "--output-dir", feature_dir,
            "--episodes", DISCOVERY_EPISODES, "--max-batches", str(DISCOVERY_MAX_BATCHES),
            "--device", "cuda", "--policy-dtype", "bfloat16"], feature_dir / "discovery.log")
        if rc != 0:
            raise RuntimeError(f"feature discovery exited {rc}")
        save_to_drive(feature_dir)

        # --- Calibrate the audit before trusting any verdict it gives ------------
        # Without this a flat reading cannot be told apart from a blind test. One
        # calibration per (run, target layer), because the pool it samples is that
        # run's own exercised features below that layer.
        calibrations, cal_by_key = {}, {}
        if RUN_AUDIT_CALIBRATION:
            for row in targets:
                layer = int(row["layer_index"])
                ckey = (str(row["run_dir"]), layer, float(row["tau"]))
                cal_by_key[row["feature_key"]] = ckey
                if ckey in calibrations:
                    continue
                cal_dir = batch_out / f"calibration_{row['run_dir'].name}_L{layer}_t{float(row['tau']):g}"
                rc = stream(f"Audit calibration below layer {layer} ({row['run_dir'].name})", [
                    PYTHON, "-u", "scripts/calibrate_circuit_audit.py", row["run_dir"],
                    "--target-layer", str(layer), "--tau", str(float(row["tau"])),
                    "--size", "16", "--repeats", "20", "--draws", str(VALIDATION_RANDOM_DRAWS),
                    "--output-dir", cal_dir], cal_dir / "calibration.log")
                path = cal_dir / "audit_calibration.json"
                calibrations[ckey] = json.loads(path.read_text()) if (rc == 0 and path.exists()) else None
                save_to_drive(cal_dir)

        # --- Trace and audit each target -----------------------------------------
        # Twenty traces is several hours. The summary is rerun after every target
        # and copied to Drive, so a dropped runtime leaves a readable partial
        # result rather than a directory of logs.
        cal_args = []
        for (run_key, layer, tau), cal in calibrations.items():
            if cal:
                cal_args += ["--calibration",
                             str(batch_out / f"calibration_{Path(run_key).name}_L{layer}_t{tau:g}")]

        def summarize(paths, *, quiet=True):
            rc = stream("Summarise H3", [
                PYTHON, "-u", "scripts/summarize_h3_rate.py", *[str(q) for q in paths], *cal_args,
                "--source-policy", TRACE_SOURCE_POLICY, "--output-dir", batch_out],
                batch_out / "summarize.log")
            if rc != 0:
                raise RuntimeError(f"summarize_h3_rate exited {rc}")
            return json.loads((batch_out / "h3_rate.json").read_text())

        traced, h3 = [], None
        for n, row in enumerate(targets, 1):
            key, run_dir = row["feature_key"], row["run_dir"]
            trace_dir = run_dir / "circuits" / key.replace(":", "_")
            print(f"\n########## {n}/{len(targets)}  {key}  ({run_dir.name}) ##########", flush=True)
            rc = stream(f"Trace {key}", [
                PYTHON, "-u", "scripts/trace_pi05_transcoder_circuit.py",
                "--policy-path", POLICY_PATH, "--seed", str(SEED),
            "--checkpoint", TRANSCODER_CHECKPOINT, "--feature-dir", feature_dir,
                "--output-dir", trace_dir, "--target", key,
                "--source-policy", TRACE_SOURCE_POLICY,
                "--top-examples", str(TRACE_TOP_EXAMPLES),
                "--parents-per-node", str(TRACE_PARENTS_PER_NODE),
                "--max-depth", str(TRACE_MAX_DEPTH), "--max-nodes", str(TRACE_MAX_NODES),
                "--device", "cuda", "--policy-dtype", "bfloat16"], trace_dir / "trace.log")
            if rc == 0:
                rc = stream(f"Audit {key}", [
                    PYTHON, "-u", "scripts/validate_circuit_with_counterfactual.py",
                    trace_dir, run_dir, "--random-draws", str(VALIDATION_RANDOM_DRAWS)],
                    trace_dir / "validation.log")
            else:
                print(f"  trace exited {rc}; the feature may not fire in the discovery pool.")
            # Saved either way: a failed trace's log is how the failure gets diagnosed,
            # and the summary counts it as failed rather than dropping it.
            save_to_drive(trace_dir, required=False)
            traced.append(trace_dir)
            h3 = summarize(traced)
            running = h3["summary"]
            note = f", {running['failed']} failed" if running["failed"] else ""
            print(f"  running total: {running['corroborated']}/{running['audited']} "
                  f"corroborated{note}", flush=True)
            save_to_drive(batch_out, required=False)

        # --- Report ----------------------------------------------------------------
        # The rate, its interval and the verdict come from a script with its own
        # tests and are read back from the artefact. Nothing here re-derives them.
        summary, audits = h3["summary"], h3["audits"]

        def _f(x, spec=".3f"):
            return "n/a" if x is None else format(float(x), spec)

        if calibrations:
            display(Markdown("### What the audit can detect"))
            rows = ""
            for ckey, cal in calibrations.items():
                if not cal:
                    rows += f"<tr><td colspan=4><i>calibration failed for {Path(ckey[0]).name} L{ckey[1]}</i></td></tr>"
                    continue
                for r in cal["levels"]:
                    rows += (f"<tr><td>{Path(ckey[0]).name} L{ckey[1]}</td>"
                             f"<td>{r['contamination']:.0%}</td>"
                             f"<td>{'inf' if r['enrichment_median'] is None else format(r['enrichment_median'], '.2f')}</td>"
                             f"<td>{r['detection_rate']:.0%}</td></tr>")
            display(HTML("<table><tr><th>pool</th><th>content</th><th>sel enrich (median)</th>"
                         "<th>detected</th></tr>" + rows + "</table>"))
            if summary["blind_pools"]:
                display(Markdown(f"**The audit is blind on {len(summary['blind_pools'])} pool(s); "
                                 "no verdict below means anything.**"))
            elif summary["detection_floor"] is not None:
                display(Markdown(
                    f"**Detection floor: the audit finds a set that is at least "
                    f"{summary['detection_floor']:.0%} content-carrying. "
                    f"A circuit it reads as flat is below that.**"))

        display(Markdown(f"### H3 over {summary['audited']} audited circuits"))
        def _canon(key):
            parts = str(key).split(":")
            if parts and parts[0][:1].upper() == "L" and parts[0][1:].isdigit():
                parts[0] = f"L{int(parts[0][1:]):02d}"
            return ":".join(parts)

        # The nominator writes L4 and the tracer writes L04; joining on the raw
        # string blanked every single-digit layer in the table below.
        sel_by_key = {_canon(r["feature_key"]): float(r["selectivity"]) for r in targets}
        task_by_key = {_canon(r["feature_key"]): r["run_dir"].name for r in targets}
        display(HTML(
            "<table><tr><th>target</th><th>task</th><th>feature sel</th><th>parents</th>"
            "<th>exercised</th><th>head n</th><th>head sel enrich</th><th>p</th>"
            "<th>full</th><th>spec</th><th>verdict</th></tr>"
            + "".join(
                f"<tr><td>{r.get('target') or r['path'][-30:]}</td>"
                f"<td>{task_by_key.get(r.get('target'), '')}</td>"
                f"<td>{_f(sel_by_key.get(r.get('target')), '.1f')}</td><td>{r.get('parents') or ''}</td>"
                f"<td>{_f(r.get('exercised'), '.1%')}</td><td>{r.get('head_nodes') or ''}</td>"
                f"<td>{_f(r.get('head_enrichment'))}</td><td>{_f(r.get('head_p'), '.4f')}</td>"
                f"<td>{_f(r.get('full_enrichment'))}</td><td>{_f(r.get('specificity'), '.2f')}</td>"
                f"<td>{str(r.get('verdict') or r['status'])[:38]}</td></tr>" for r in audits)
            + "</table>"
        ))
        ci = summary["clopper_pearson_95"]
        if summary["audited"]:
            line = (f"**H3: {summary['corroborated']} of {summary['audited']} circuits corroborated "
                    f"= {summary['rate']:.0%}, 95% CI [{ci['low']:.1%}, {ci['high']:.1%}]**")
            if summary["reading"]:
                line += f"\n\n{summary['reading']}."
            display(Markdown(line))
        display(Markdown(f"**Verdict: {summary['verdict']}**"))

        save_to_drive(batch_out)
        write_manifest(h3=summary)
        print(f"\nH3 artefacts: {batch_out}")
        if SAVE_RUNS_TO_DRIVE:
            print("On Drive:", DRIVE_BATCH / "h3")
        print("Paste the detection floor and the H3 rate into the write-up.")

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Trace']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


## New transcoder

In [ ]:
# @title Select new checkpoint
ARM='new'
TRANSCODER_DRIVE_PATH=CHECKPOINTS[ARM]
TRANSCODER_CHECKPOINT=LOCAL_CHECKPOINTS[ARM]
RESULTS[ARM]={'errors':{}}
run_dirs=[]


In [ ]:
# @title New: Fidelity
import traceback, shutil
try:
    fidelity_dir=COMPARISON_ROOT/ARM/'fidelity'
    run([PYTHON,'-u',LOCAL_REPO/'scripts/compare_transcoder_fidelity.py',
         '--checkpoint',TRANSCODER_CHECKPOINT,'--output-dir',fidelity_dir,
         '--tasks',','.join(map(str,TASK_ID_LIST)), '--starts',FIDELITY_STARTS,
         '--policy-path',POLICY_PATH,'--suite',SUITE,'--seed',str(SEED)])

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Fidelity']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title New: Probe
import traceback, shutil
try:
    # @title Run Counterfactual Probe

    import json
    import os
    import subprocess
    import sys
    import time
    from pathlib import Path
    from IPython.display import display, HTML, Markdown, Image as IPyImage

    batch_dir = COMPARISON_ROOT / ARM / "probe"
    batch_dir.mkdir(parents=True, exist_ok=True)


    import shutil

    DRIVE_BATCH = (DRIVE_ROOT / batch_dir.relative_to(LOCAL_REPO)) if SAVE_RUNS_TO_DRIVE else None


    def _tree_stats(root: Path) -> tuple[int, int]:
        files = [p for p in root.rglob("*") if p.is_file()]
        return len(files), sum(p.stat().st_size for p in files)


    def save_to_drive(local_dir: Path, *, required: bool = True) -> Path | None:
        """Mirror one directory under DRIVE_ROOT/outputs/, preserving its path.

        Colab's disk does not survive the runtime, and neither does a disconnect
        four hours into a twenty-run loop. Every artefact the paper's tables are
        filled from (the delta store, decision_metrics.json, validation.json,
        provenance.json, the traced graph, the audit calibration) lives in these
        directories, so each is copied the moment it is complete rather than at the
        end of the notebook.

        The copy is verified. A Drive FUSE mount that has run out of quota returns
        success on write and leaves a short file, so comparing the byte count is
        the only way to know the data is really there.
        """
        if not SAVE_RUNS_TO_DRIVE:
            return None
        local_dir = Path(local_dir)
        if not local_dir.exists():
            return None
        rel = local_dir.relative_to(LOCAL_REPO)
        target = DRIVE_ROOT / rel
        src_files, src_bytes = _tree_stats(local_dir)
        try:
            shutil.copytree(local_dir, target, dirs_exist_ok=True)
            dst_files, dst_bytes = _tree_stats(target)
            short = src_bytes - dst_bytes
            if dst_files < src_files or short > 0:
                raise IOError(
                    f"copy is short by {src_files - dst_files} files / {short / 1024**2:.1f} MB "
                    f"(Drive quota or a dropped mount)"
                )
        except Exception as exc:
            message = f"!! FAILED to save {rel} to Drive: {exc}"
            if required:
                raise RuntimeError(
                    message + f"\n   The data is still on local disk at {local_dir}. "
                    "Free space or remount, then re-run this cell's save."
                ) from exc
            print(message, flush=True)
            return None
        print(f"saved {rel} -> {target}  ({dst_files} files, {dst_bytes / 1024**2:.0f} MB)", flush=True)
        return target


    def _log_tail(path: Path, lines: int = 20) -> list[str]:
        """The end of a failed run's log, carried in the manifest.

        A return code says a run failed and nothing about why. The log directory is
        saved too, but the manifest is the file someone reads first.
        """
        try:
            return [line[-300:] for line in
                    path.read_text(errors="replace").splitlines()[-lines:]]
        except OSError:
            return []


    def write_manifest(**extra) -> None:
        """An index of the batch, written into Drive beside the runs.

        Further analysis should be able to start from the Drive folder alone,
        without the notebook or the Colab session that produced it.
        """
        if not SAVE_RUNS_TO_DRIVE:
            return
        manifest = {
            "batch": batch_dir.name,
            "written_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "drive_root": str(DRIVE_ROOT),
            "config": {
                "suite": SUITE, "task_ids": TASK_ID_LIST, "seeds": SEED_LIST,
                "states": STATES, "state_stride": STATE_STRIDE, "noise_samples": NOISE_SAMPLES,
                "perturbation": PERTURBATION, "color": COLOR, "dose": DOSE,
                "policy_path": POLICY_PATH, "transcoder_checkpoint": TRANSCODER_DRIVE_PATH,
                "trace_n_targets": TRACE_N_TARGETS, "trace_source_policy": TRACE_SOURCE_POLICY,
            },
            "runs": [],
            "failures": [
                {"task_id": t, "seed": s, "returncode": rc,
                 "dir": f"task{t:02d}_seed{s}",
                 "log_tail": _log_tail(batch_dir / f"task{t:02d}_seed{s}" / "probe.log")}
                for t, s, rc in failures
            ],
            **extra,
            "reading": {
                "per task": "task*/decision_metrics.json -> h1, h2",
                "pooled over tasks": "aggregate/aggregate.json -> summary",
                "delta store": "task*/latents/ (signed per-feature deltas, npz blocks)",
                "H3 rate": "h3/h3_rate.json -> summary",
                "audit sensitivity": "h3/calibration_*/audit_calibration.json -> summary",
            },
        }
        for run_dir in run_dirs:
            entry = {"dir": run_dir.name}
            for name, key in (("decision_metrics.json", "verdicts"),
                              ("provenance.json", "provenance"),
                              ("nominated_targets.json", "nominated")):
                path = run_dir / name
                if not path.exists():
                    continue
                data = json.loads(path.read_text())
                if key == "verdicts":
                    entry["h1"] = data["h1"]["verdict"]
                    entry["h2"] = data["h2"]["verdict"]
                    entry["sel_adj_l2"] = data["h1"]["pooled"].get("sel_adj_l2")
                elif key == "provenance":
                    entry["commit"] = data.get("git", {}).get("commit")
                    entry["dirty"] = data.get("git", {}).get("dirty")
                else:
                    entry["nominated"] = [r["feature_key"] for r in data.get("nominated") or []]
            entry["files"], entry["bytes"] = _tree_stats(run_dir)
            manifest["runs"].append(entry)
        (batch_dir / "manifest.json").write_text(json.dumps(manifest, indent=2))
        target = DRIVE_BATCH / "manifest.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(manifest, indent=2))
        print(f"manifest -> {target}", flush=True)

    def build_cmd(task_id, seed, run_dir):
        cmd = [
            str(PYTHON), "-u", "scripts/probe_pi05_transcoder_counterfactual.py",
            "--policy-path", POLICY_PATH,
            "--output-dir", str(run_dir),
            "--suite", SUITE,
            "--task-id", str(task_id),
            "--seed", str(seed),
            "--device", "cuda",
            "--policy-dtype", "bfloat16",
        ]
        if LIST_OBJECTS_ONLY:
            cmd.append("--list-objects")
            return cmd
        cmd += [
            "--checkpoint", TRANSCODER_CHECKPOINT,
            "--perturbation", PERTURBATION,
            "--color", COLOR,
            "--dose", DOSE,
            "--states", str(STATES),
            "--state-stride", str(STATE_STRIDE),
            "--noise-samples", str(NOISE_SAMPLES),
            "--noise-seed", str(seed),
        ]
        # Omitted flags let each task pick its own object of interest and sibling.
        if TARGET.strip():
            cmd += ["--target", TARGET.strip()]
        if PLACEBO_TARGET.strip():
            cmd += ["--placebo-target", PLACEBO_TARGET.strip()]
        return cmd


    env = os.environ.copy()
    env["PYTHONUNBUFFERED"] = "1"
    # Colab's inline matplotlib backend is only valid inside the kernel process;
    # inheriting it breaks LIBERO's import. The script guards this too.
    env["MPLBACKEND"] = "Agg"
    env["MUJOCO_GL"] = "egl"
    env["PYOPENGL_PLATFORM"] = "egl"
    env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
        os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else ""
    )


    def emit(chunk: bytes) -> None:
        """Write through to the cell output.

        Colab's sys.stdout is an ipykernel OutStream with no .buffer, so decode
        rather than assuming a binary stream exists.
        """
        stream = getattr(sys.stdout, "buffer", None)
        if stream is not None:
            stream.write(chunk)
        else:
            sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()


    def run_probe(task_id, seed):
        run_dir = batch_dir / f"task{task_id:02d}_seed{seed}"
        run_dir.mkdir(parents=True, exist_ok=True)
        cmd = build_cmd(task_id, seed, run_dir)
        log_path = run_dir / "probe.log"
        print(f"\n{'=' * 78}\n== task {task_id}, seed {seed}\n$ {' '.join(cmd)}\n", flush=True)
        with log_path.open("wb") as log_handle:
            process = subprocess.Popen(cmd, cwd=LOCAL_REPO, env=env,
                                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
            try:
                while True:
                    chunk = process.stdout.read(4096)
                    if not chunk:
                        break
                    log_handle.write(chunk); log_handle.flush()
                    emit(chunk)
                rc = process.wait()
            except BaseException:
                # Never leave the probe running headless if the cell is interrupted.
                process.kill(); process.wait(); raise
        return rc, run_dir, log_path


    # One task failing must not discard the tasks that already ran: this is an
    # hour-long loop and a mid-run abort would cost all of it.
    started = time.time()
    run_dirs, failures = [], []
    plan = [(t, s) for t in TASK_ID_LIST for s in SEED_LIST]
    for index, (task_id, seed) in enumerate(plan, start=1):
        print(f"\n[{index}/{len(plan)}]  elapsed {(time.time() - started) / 60:.1f} min", flush=True)
        rc, run_dir, log_path = run_probe(task_id, seed)
        # Saved either way, and immediately rather than at the end of the loop. A
        # disconnect twelve runs in would otherwise take all twelve, and a failed
        # run's log is the only record of why it failed: on the first full run
        # task 5 failed and its log died with the runtime.
        save_to_drive(run_dir, required=False)
        if rc == 0:
            run_dirs.append(run_dir)
        else:
            failures.append((task_id, seed, rc))
            print(f"\n!! task {task_id} seed {seed} exited {rc}; continuing. Tail of {log_path}:", flush=True)
            for line in log_path.read_text(errors="replace").splitlines()[-12:]:
                print("   ", line[-300:], flush=True)

    elapsed = time.time() - started
    print(f"\n{'=' * 78}")
    print(f"{len(run_dirs)} of {len(plan)} runs completed in {elapsed / 60:.1f} min")
    if failures:
        print("failed:", failures)
    if not run_dirs:
        raise SystemExit("No task produced a run; nothing to aggregate.")
    out_dir = run_dirs[0]  # the trace stage below works from a single task's run

    if LIST_OBJECTS_ONLY:
        display(Markdown(
            "### Objects listed\n\nSet `LIST_OBJECTS_ONLY = False` to run the measurement. "
            "Override `TARGET` / `PLACEBO_TARGET` only if you do not want the task's own objects."
        ))
    else:
        display(Markdown("### Per-task verdicts"))
        rows = []
        for run_dir in run_dirs:
            try:
                d = json.loads((run_dir / "decision_metrics.json").read_text())
            except FileNotFoundError:
                continue
            h1, h2 = d["h1"], d["h2"]
            rows.append(
                f"<tr><td>{run_dir.name}</td>"
                f"<td>{len(h1['cells'])}</td>"
                f"<td>{h1['pooled']['sel_raw']:.2f}</td>"
                f"<td>{(h1['pooled']['sel_adj_l2'] or float('nan')):.2f}</td>"
                f"<td>{h1['verdict'][:28]}</td>"
                f"<td>{(h2.get('referent_check') or {}).get('referent_moved')}</td>"
                f"<td>{h2['verdict'][:34]}</td></tr>"
            )
        display(HTML(
            "<table><tr><th>run</th><th>cells</th><th>Sel raw</th><th>Sel adj</th>"
            "<th>H1</th><th>referent moved</th><th>H2</th></tr>" + "".join(rows) + "</table>"
        ))
        print("Single-task verdicts are descriptive. The pooled verdict is in the next cell.")

        shapes_path = out_dir / "observation_shapes.json"
        if shapes_path.exists():
            shapes = json.loads(shapes_path.read_text())
            display(Markdown("### Observation Shapes Sent To The Policy"))
            display(HTML(
                "<table><tr><th>key</th><th>shape</th><th>dtype</th></tr>"
                + "".join(f"<tr><td>{k}</td><td>{v['shape']}</td><td>{v['dtype']}</td></tr>"
                          for k, v in sorted(shapes.get("tensor_shapes", {}).items()))
                + f"<tr><td>noise</td><td>{shapes.get('noise_shape')}</td>"
                  f"<td>{shapes.get('noise_dtype')}</td></tr></table>"
            ))
            print("task:", repr(shapes.get("task")))

        decision_path = out_dir / "decision_metrics.json"
        if decision_path.exists():
            decision = json.loads(decision_path.read_text())
            h1, h2 = decision["h1"], decision["h2"]

            def _f(v, spec=".4g"):
                return "n/a" if v is None else format(float(v), spec)

            pooled, spread, ci = h1["pooled"], h1["spread"], h1["bootstrap_ci_95"]
            display(Markdown(f"### H1 -- object selectivity (task prompt, {len(h1['cells'])} cells)"))
            display(HTML(
                "<table><tr><th>quantity</th><th>target</th><th>placebo</th><th>ratio</th></tr>"
                f"<tr><td>D, layer response (latent L2)</td><td>{_f(pooled['D_target'])}</td><td>{_f(pooled['D_placebo'])}</td><td>{_f(pooled['sel_raw'])}</td></tr>"
                f"<tr><td>S, changed-pixel fraction</td><td>{_f(pooled['S_px_target'])}</td><td>{_f(pooled['S_px_placebo'])}</td><td>{_f(pooled['S_px_ratio'])}</td></tr>"
                f"<tr><td>S, image-difference norm</td><td>{_f(pooled['S_l2_target'])}</td><td>{_f(pooled['S_l2_placebo'])}</td><td>{_f(pooled['S_l2_ratio'])}</td></tr>"
                f"<tr><td>action relative L2</td><td>{_f(pooled['A_target'])}</td><td>{_f(pooled['A_placebo'])}</td><td>{_f(pooled['action_sel'])}</td></tr>"
                "</table>"
            ))
            display(HTML(
                "<table><tr><th>selectivity</th><th>pooled</th><th>cells mean ± sd</th><th>min .. max</th><th>95% CI</th></tr>"
                + "".join(
                    f"<tr><td>{label}</td><td>{_f(pooled[k])}</td><td>{_f(spread[k]['mean'])} ± {_f(spread[k]['sd'])}</td>"
                    f"<td>{_f(spread[k]['min'])} .. {_f(spread[k]['max'])}</td><td>[{_f(ci[k]['low'])}, {_f(ci[k]['high'])}]</td></tr>"
                    for k, label in (("sel_raw", "raw"), ("sel_adj_px", "adjusted, |P|"), ("sel_adj_l2", "adjusted, ||dI||"))
                )
                + "</table>"
            ))
            nf = h1["null_floor"]
            print(f"null floor D: mean={_f(nf['mean'])} max={_f(nf['max'])} -> {'near zero' if h1['null_floor_near_zero'] else 'NOT near zero'}")
            pc = h1.get("positive_control", {})
            if pc.get("n"):
                print(f"positive control (prompt swap) D={_f(pc['D'])} (n={pc['n']}): target is {_f(pc['target_over_positive'])} of it, "
                      f"placebo {_f(pc['placebo_over_positive'])}")
            cs = h1.get("cluster_spread", {})
            if cs.get("n"):
                sp = cs["sel_adj_l2"]
                print(f"clusters {cs['unit']}: n={cs['n']}  Sel adj ||dI|| min={_f(sp['min'])} max={_f(sp['max'])}")
            rb = h1.get("robustness", {})
            if rb:
                print(f"robustness: Sel rel={_f(rb['sel_rel'])}  per-layer geomean={_f(rb['sel_layer_geomean'])} "
                      f"[{_f(rb['sel_layer_min'])} .. {_f(rb['sel_layer_max'])}], {rb['layers_above_one']}/{rb['layers']} layers above 1; "
                      f"direction agrees: {rb['agree_in_direction']}")
            for kind, dr in h1["dose_response"].items():
                print(f"dose response [{kind}]: elasticity={_f(dr['elasticity_mean'])} monotone={dr['monotone_in_dose']}  "
                      + "  ".join(f"d={r['dose']:g}: D={_f(r['D'])}" for r in dr["rows"]))
            display(Markdown(f"**H1 verdict: {h1['verdict']}**  \n<small>rule: {h1['decision_rule']}</small>"))

            g = h2["grounding"]
            display(Markdown("### H2 -- referent versus position (prompt swap over identical pixels)"))
            display(HTML(
                "<table><tr><th>prompt</th><th>D target</th><th>D placebo</th><th>selectivity</th><th>cells</th></tr>"
                + "".join(
                    f"<tr><td>{p}</td><td>{_f(r['D_target'])}</td><td>{_f(r['D_placebo'])}</td><td>{_f(r['sel_raw'])}</td><td>{r['n_cells']}</td></tr>"
                    for p, r in h2["per_prompt"].items()
                )
                + "</table>"
            ))
            print(f"prompt grounding g: mean={_f(g['mean'])} min={_f(g['min'])} max={_f(g['max'])} (n={g['n']}, threshold {h2['grounding_threshold']}); "
                  f"perturbation action effect {_f(h2['perturbation_action_effect'])}")
            rc = h2.get("referent_check", {})
            if rc:
                print(f"referent check (action anchor A_target/A_placebo): task={_f(rc.get('behavioural_anchor_task'))} "
                      f"alt={_f(rc.get('behavioural_anchor_alt'))} -> referent moved: {rc.get('referent_moved')}")
            display(Markdown(f"**H2 verdict: {h2['verdict']}**  \n<small>rule: {h2['decision_rule']}</small>"))

        provenance_path = out_dir / "provenance.json"
        if provenance_path.exists():
            pv = json.loads(provenance_path.read_text())
            ck = (pv.get("transcoder_checkpoint") or {}).get("digest", "")
            print(f"provenance: commit {pv['git'].get('commit')} dirty={pv['git'].get('dirty')} "
                  f"lerobot={pv['packages'].get('lerobot')} torch={pv['packages'].get('torch')} "
                  f"device={pv['device'].get('gpu_name', pv['device'].get('device'))} checkpoint sha256={ck[:16]}")

        summary_path = out_dir / "counterfactual_summary.json"
        if summary_path.exists():
            summary = json.loads(summary_path.read_text())
            display(Markdown("### Counterfactual Activation Response (every measurement)"))
            display(HTML(
                "<table><tr><th>state</th><th>noise</th><th>prompt</th><th>kind</th><th>target</th><th>dose</th>"
                "<th>changed pixels</th><th>latent L2</th><th>action rel L2</th></tr>"
                + "".join(
                    "<tr><td>{s}</td><td>{n}</td><td>{p}</td><td>{k}</td><td>{t}</td><td>{d}</td><td>{px:.5f}</td>"
                    "<td>{lat}</td><td>{act:.5g}</td></tr>".format(
                        s=r["state_index"], n=r.get("noise_index", 0), p=r.get("prompt", "task"),
                        k=r["kind"], t=r["target"], d=r["dose"],
                        px=r["pixel"]["changed_pixel_fraction"],
                        lat=("n/a" if r["latent"].get("l2_delta_mean") is None
                             else "{:.5g}".format(r["latent"]["l2_delta_mean"])),
                        act=r["action_relative_l2"])
                    for r in summary.get("measurements", []))
                + "</table>"
            ))
            print("Read every row against the 'null' rows: that is the nondeterminism floor.")

        # One state's frames for the eye: every camera's baseline, the target and
        # placebo recolours at the top dose, and their difference images.
        pngs = sorted((out_dir / "images").glob("state0_*.png"))
        keep = [p for p in pngs if "baseline" in p.name or "dose1_" in p.name]
        for png in keep[:16]:
            display(Markdown(f"**{png.name}**"))
            display(IPyImage(filename=str(png)))

        # Rank features by selectivity: responds to the task object, not to the
        # visually matched control. Run on EVERY task, not just the first: the H3
        # rate is supposed to be pooled across tasks, and nominating from one run
        # makes it a rate over one scene no matter how the trace stage samples.
        nominated_counts = {}
        for index, run_dir in enumerate(run_dirs, start=1):
            report_cmd = [str(PYTHON), "-u", "scripts/report_pi05_counterfactual_features.py",
                          str(run_dir), "--top", "30",
                          "--min-consistency", str(MIN_TRACE_CONSISTENCY),
                          "--min-trace-layer", str(MIN_TRACE_LAYER),
                          "--min-selectivity", str(MIN_SELECTIVITY)]
            report = subprocess.run(report_cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
            path = run_dir / "nominated_targets.json"
            n = len(json.loads(path.read_text()).get("nominated") or []) if path.exists() else 0
            nominated_counts[run_dir.name] = n
            if index == 1:
                print("\n$", " ".join(report_cmd), flush=True)
                print(report.stdout)          # the full ranking, once, for the eye
            if report.returncode != 0:
                print(f"  !! report on {run_dir.name} exited {report.returncode}")
                print(report.stderr[-800:])
        total_nominees = sum(nominated_counts.values())
        print(f"\nnominees per run: " +
              "  ".join(f"{k.replace('task', 't')}:{v}" for k, v in nominated_counts.items()))
        print(f"{total_nominees} nominees across {len(run_dirs)} runs, "
              f"{sum(1 for v in nominated_counts.values() if v)} runs contributing.")
        if sum(1 for v in nominated_counts.values() if v) < 2:
            print("  WARNING: fewer than two runs nominated anything, so the H3 rate "
                  "cannot be a rate over tasks. Relax MIN_SELECTIVITY or MIN_TRACE_LAYER.")

    # Every run was copied as it finished; this re-copies whatever the report stage
    # added afterwards (candidate_features.csv, nominated_targets.json) and writes
    # the batch index.
    for run_dir in run_dirs:
        save_to_drive(run_dir)
    write_manifest()
    print("\nArtifacts:", batch_dir)
    if SAVE_RUNS_TO_DRIVE:
        print("On Drive:", DRIVE_BATCH)

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Probe']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title New: Aggregate
import traceback, shutil
try:
    if not run_dirs: raise RuntimeError('Probe produced no successful runs')
    # @title Aggregate Across Tasks

    import json, os, subprocess, sys
    from pathlib import Path
    from IPython.display import display, HTML, Markdown

    agg_dir = batch_dir / "aggregate"
    cmd = [str(PYTHON), "-u", "scripts/aggregate_counterfactual_runs.py", *[str(d) for d in run_dirs],
           "--output-dir", str(agg_dir)]
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, cwd=LOCAL_REPO, env=env, capture_output=True, text=True)
    print(proc.stdout)
    if proc.returncode != 0:
        print(proc.stderr[-3000:]); raise SystemExit(f"aggregation exited {proc.returncode}")

    report = json.loads((agg_dir / "aggregate.json").read_text())
    s = report["summary"]

    def _f(v, spec=".2f"):
        return "n/a" if v is None else format(float(v), spec)

    display(Markdown(f"### H1 pooled over {s['tasks']} tasks ({s['cells_total']} cells)"))
    p, ci = s["pooled"], s["bootstrap_ci_95"]
    display(HTML(
        "<table><tr><th>quantity</th><th>pooled</th><th>task-cluster 95% CI</th></tr>"
        + "".join(
            f"<tr><td>{label}</td><td>{_f(p.get(k))}</td>"
            f"<td>[{_f(ci[k].get('low'))}, {_f(ci[k].get('high'))}]</td></tr>"
            for k, label in (("sel_raw", "raw selectivity"),
                             ("sel_adj_l2", "adjusted, image norm (decides)"),
                             ("sel_adj_px", "adjusted, pixel count")))
        + "</table>"
    ))
    print(f"tasks with adjusted selectivity above 1: {s['tasks_above_one']} of {s['tasks']}")

    display(Markdown("**Per task** (the spread here is what the pilot could not see)"))
    display(HTML(
        "<table><tr><th>task</th><th>cells</th><th>Sel raw</th><th>Sel adj</th><th>action sel</th></tr>"
        + "".join(
            f"<tr><td>{r['task_id']}</td><td>{r['n_cells']}</td><td>{_f(r['sel_raw'])}</td>"
            f"<td>{_f(r['sel_adj_l2'])}</td><td>{_f(r['action_sel'])}</td></tr>"
            for r in report["per_task"])
        + "</table>"
    ))
    display(Markdown(f"**H1 verdict: {s['verdict']}**  \n<small>rule: {s['decision_rule']}</small>"))

    display(Markdown("### H2, gated per task"))
    display(HTML(
        "<table><tr><th>task</th><th>g</th><th>anchor task</th><th>anchor alt</th>"
        "<th>referent moved</th><th>Sel task</th><th>Sel alt</th></tr>"
        + "".join(
            f"<tr><td>{r['task_id']}</td><td>{_f(r['grounding'],'.3f')}</td><td>{_f(r['anchor_task'])}</td>"
            f"<td>{_f(r['anchor_alt'])}</td><td><b>{r['referent_moved']}</b></td>"
            f"<td>{_f(r['sel_task'])}</td><td>{_f(r['sel_alt'])}</td></tr>"
            for r in report["h2_per_task"])
        + "</table>"
    ))
    display(Markdown(f"**H2 verdict: {s['h2_verdict']}**"))

    save_to_drive(agg_dir)
    print("\nPaste the two verdict lines and the per-task tables into the write-up.")

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Aggregate']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title New: Trace
import traceback, shutil
try:
    if not run_dirs: raise RuntimeError('Probe produced no successful runs')
    # @title Trace And Audit Several Targets

    import csv, json, math, os, subprocess, sys, time
    from pathlib import Path
    from IPython.display import display, HTML, Markdown

    if not RUN_CIRCUIT_TRACE:
        print("RUN_CIRCUIT_TRACE is off; skipping.")
    else:
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"; env["MPLBACKEND"] = "Agg"
        env["MUJOCO_GL"] = "egl"; env["PYOPENGL_PLATFORM"] = "egl"
        env["PYTHONPATH"] = str(LOCAL_REPO / "src") + (
            os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

        def stream(label, cmd, log_path):
            print(f"\n== {label} ==\n$ {' '.join(map(str, cmd))}", flush=True)
            log_path.parent.mkdir(parents=True, exist_ok=True)
            with log_path.open("wb") as handle:
                proc = subprocess.Popen([str(c) for c in cmd], cwd=LOCAL_REPO, env=env,
                                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
                try:
                    while True:
                        chunk = proc.stdout.read(4096)
                        if not chunk:
                            break
                        handle.write(chunk); handle.flush()
                        out = getattr(sys.stdout, "buffer", None)
                        (out.write(chunk) if out is not None else
                         sys.stdout.write(chunk.decode("utf-8", "replace")))
                        sys.stdout.flush()
                    return proc.wait()
                except BaseException:
                    proc.kill(); proc.wait(); raise

        # --- Pool nominees across every task -------------------------------------
        # A rate computed from one task's nominees is a rate over that scene. Each
        # nominee keeps the run it came from, because its audit has to read the
        # delta store that measured it.
        by_run, seen = {}, set()
        for run_dir in run_dirs:
            path = run_dir / "nominated_targets.json"
            if not path.exists():
                continue
            rows = []
            for row in json.loads(path.read_text()).get("nominated") or []:
                key = row["feature_key"]
                if key in seen:      # the same feature nominated by two tasks is
                    continue         # one circuit, and would be traced once anyway
                seen.add(key)
                rows.append({"run_dir": run_dir, **row})
            if rows:
                rows.sort(key=lambda r: -float(r["target_z"]))
                by_run[run_dir] = rows
        if not by_run:
            raise RuntimeError(
                "No task nominated a target under the rule. Widen the probe (STATES, SEEDS, DOSE) "
                "or relax MIN_TRACE_CONSISTENCY / MIN_TRACE_LAYER / MIN_SELECTIVITY deliberately."
            )

        # Round-robin across tasks, strongest nominee of each first. Sorting the
        # pooled nominees by z instead lets the single most selective task take
        # every slot, which is what happened on the first run: all five targets
        # came from one scene and the "rate over the method" was a rate over that
        # scene. Round-robin makes single-task monopoly impossible whenever more
        # than one task nominated anything.
        targets, depth = [], 0
        while len(targets) < TRACE_N_TARGETS:
            layer = [rows[depth] for rows in by_run.values() if len(rows) > depth]
            if not layer:
                break
            layer.sort(key=lambda r: -float(r["target_z"]))
            targets.extend(layer[: TRACE_N_TARGETS - len(targets)])
            depth += 1
        pool = [row for rows in by_run.values() for row in rows]
        tasks_represented = len({str(r["run_dir"]) for r in targets})
        if tasks_represented < 2:
            print("  WARNING: every target comes from one task, so the rate below is a rate "
                  "over that scene and not over the method.")
        bound = 1 - 0.05 ** (1 / len(targets))
        print(f"{len(pool)} distinct nominees from {len(by_run)} of {len(run_dirs)} runs; "
              f"tracing {len(targets)} across {tasks_represented} task(s).")
        print(f"A null result over {len(targets)} bounds the corroboration rate below {bound:.1%}.\n")
        for row in targets:
            print(f"  {row['feature_key']:<22} z={float(row['target_z']):+.2f}  "
                  f"sel={float(row['selectivity']):.1f}x  from {row['run_dir'].name}")

        batch_out = batch_dir / "h3"
        batch_out.mkdir(parents=True, exist_ok=True)

        # --- Discovery once, shared by every trace -------------------------------
        stamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
        feature_dir = LOCAL_REPO / "outputs/features/pi05_libero" / f"comparison-{RUN_ID}-{ARM}"
        rc = stream("Feature discovery", [
            PYTHON, "-u", "scripts/collect_pi05_transcoder_features.py",
            "--policy-path", POLICY_PATH, "--seed", str(SEED),
            "--checkpoint", TRANSCODER_CHECKPOINT, "--output-dir", feature_dir,
            "--episodes", DISCOVERY_EPISODES, "--max-batches", str(DISCOVERY_MAX_BATCHES),
            "--device", "cuda", "--policy-dtype", "bfloat16"], feature_dir / "discovery.log")
        if rc != 0:
            raise RuntimeError(f"feature discovery exited {rc}")
        save_to_drive(feature_dir)

        # --- Calibrate the audit before trusting any verdict it gives ------------
        # Without this a flat reading cannot be told apart from a blind test. One
        # calibration per (run, target layer), because the pool it samples is that
        # run's own exercised features below that layer.
        calibrations, cal_by_key = {}, {}
        if RUN_AUDIT_CALIBRATION:
            for row in targets:
                layer = int(row["layer_index"])
                ckey = (str(row["run_dir"]), layer, float(row["tau"]))
                cal_by_key[row["feature_key"]] = ckey
                if ckey in calibrations:
                    continue
                cal_dir = batch_out / f"calibration_{row['run_dir'].name}_L{layer}_t{float(row['tau']):g}"
                rc = stream(f"Audit calibration below layer {layer} ({row['run_dir'].name})", [
                    PYTHON, "-u", "scripts/calibrate_circuit_audit.py", row["run_dir"],
                    "--target-layer", str(layer), "--tau", str(float(row["tau"])),
                    "--size", "16", "--repeats", "20", "--draws", str(VALIDATION_RANDOM_DRAWS),
                    "--output-dir", cal_dir], cal_dir / "calibration.log")
                path = cal_dir / "audit_calibration.json"
                calibrations[ckey] = json.loads(path.read_text()) if (rc == 0 and path.exists()) else None
                save_to_drive(cal_dir)

        # --- Trace and audit each target -----------------------------------------
        # Twenty traces is several hours. The summary is rerun after every target
        # and copied to Drive, so a dropped runtime leaves a readable partial
        # result rather than a directory of logs.
        cal_args = []
        for (run_key, layer, tau), cal in calibrations.items():
            if cal:
                cal_args += ["--calibration",
                             str(batch_out / f"calibration_{Path(run_key).name}_L{layer}_t{tau:g}")]

        def summarize(paths, *, quiet=True):
            rc = stream("Summarise H3", [
                PYTHON, "-u", "scripts/summarize_h3_rate.py", *[str(q) for q in paths], *cal_args,
                "--source-policy", TRACE_SOURCE_POLICY, "--output-dir", batch_out],
                batch_out / "summarize.log")
            if rc != 0:
                raise RuntimeError(f"summarize_h3_rate exited {rc}")
            return json.loads((batch_out / "h3_rate.json").read_text())

        traced, h3 = [], None
        for n, row in enumerate(targets, 1):
            key, run_dir = row["feature_key"], row["run_dir"]
            trace_dir = run_dir / "circuits" / key.replace(":", "_")
            print(f"\n########## {n}/{len(targets)}  {key}  ({run_dir.name}) ##########", flush=True)
            rc = stream(f"Trace {key}", [
                PYTHON, "-u", "scripts/trace_pi05_transcoder_circuit.py",
                "--policy-path", POLICY_PATH, "--seed", str(SEED),
            "--checkpoint", TRANSCODER_CHECKPOINT, "--feature-dir", feature_dir,
                "--output-dir", trace_dir, "--target", key,
                "--source-policy", TRACE_SOURCE_POLICY,
                "--top-examples", str(TRACE_TOP_EXAMPLES),
                "--parents-per-node", str(TRACE_PARENTS_PER_NODE),
                "--max-depth", str(TRACE_MAX_DEPTH), "--max-nodes", str(TRACE_MAX_NODES),
                "--device", "cuda", "--policy-dtype", "bfloat16"], trace_dir / "trace.log")
            if rc == 0:
                rc = stream(f"Audit {key}", [
                    PYTHON, "-u", "scripts/validate_circuit_with_counterfactual.py",
                    trace_dir, run_dir, "--random-draws", str(VALIDATION_RANDOM_DRAWS)],
                    trace_dir / "validation.log")
            else:
                print(f"  trace exited {rc}; the feature may not fire in the discovery pool.")
            # Saved either way: a failed trace's log is how the failure gets diagnosed,
            # and the summary counts it as failed rather than dropping it.
            save_to_drive(trace_dir, required=False)
            traced.append(trace_dir)
            h3 = summarize(traced)
            running = h3["summary"]
            note = f", {running['failed']} failed" if running["failed"] else ""
            print(f"  running total: {running['corroborated']}/{running['audited']} "
                  f"corroborated{note}", flush=True)
            save_to_drive(batch_out, required=False)

        # --- Report ----------------------------------------------------------------
        # The rate, its interval and the verdict come from a script with its own
        # tests and are read back from the artefact. Nothing here re-derives them.
        summary, audits = h3["summary"], h3["audits"]

        def _f(x, spec=".3f"):
            return "n/a" if x is None else format(float(x), spec)

        if calibrations:
            display(Markdown("### What the audit can detect"))
            rows = ""
            for ckey, cal in calibrations.items():
                if not cal:
                    rows += f"<tr><td colspan=4><i>calibration failed for {Path(ckey[0]).name} L{ckey[1]}</i></td></tr>"
                    continue
                for r in cal["levels"]:
                    rows += (f"<tr><td>{Path(ckey[0]).name} L{ckey[1]}</td>"
                             f"<td>{r['contamination']:.0%}</td>"
                             f"<td>{'inf' if r['enrichment_median'] is None else format(r['enrichment_median'], '.2f')}</td>"
                             f"<td>{r['detection_rate']:.0%}</td></tr>")
            display(HTML("<table><tr><th>pool</th><th>content</th><th>sel enrich (median)</th>"
                         "<th>detected</th></tr>" + rows + "</table>"))
            if summary["blind_pools"]:
                display(Markdown(f"**The audit is blind on {len(summary['blind_pools'])} pool(s); "
                                 "no verdict below means anything.**"))
            elif summary["detection_floor"] is not None:
                display(Markdown(
                    f"**Detection floor: the audit finds a set that is at least "
                    f"{summary['detection_floor']:.0%} content-carrying. "
                    f"A circuit it reads as flat is below that.**"))

        display(Markdown(f"### H3 over {summary['audited']} audited circuits"))
        def _canon(key):
            parts = str(key).split(":")
            if parts and parts[0][:1].upper() == "L" and parts[0][1:].isdigit():
                parts[0] = f"L{int(parts[0][1:]):02d}"
            return ":".join(parts)

        # The nominator writes L4 and the tracer writes L04; joining on the raw
        # string blanked every single-digit layer in the table below.
        sel_by_key = {_canon(r["feature_key"]): float(r["selectivity"]) for r in targets}
        task_by_key = {_canon(r["feature_key"]): r["run_dir"].name for r in targets}
        display(HTML(
            "<table><tr><th>target</th><th>task</th><th>feature sel</th><th>parents</th>"
            "<th>exercised</th><th>head n</th><th>head sel enrich</th><th>p</th>"
            "<th>full</th><th>spec</th><th>verdict</th></tr>"
            + "".join(
                f"<tr><td>{r.get('target') or r['path'][-30:]}</td>"
                f"<td>{task_by_key.get(r.get('target'), '')}</td>"
                f"<td>{_f(sel_by_key.get(r.get('target')), '.1f')}</td><td>{r.get('parents') or ''}</td>"
                f"<td>{_f(r.get('exercised'), '.1%')}</td><td>{r.get('head_nodes') or ''}</td>"
                f"<td>{_f(r.get('head_enrichment'))}</td><td>{_f(r.get('head_p'), '.4f')}</td>"
                f"<td>{_f(r.get('full_enrichment'))}</td><td>{_f(r.get('specificity'), '.2f')}</td>"
                f"<td>{str(r.get('verdict') or r['status'])[:38]}</td></tr>" for r in audits)
            + "</table>"
        ))
        ci = summary["clopper_pearson_95"]
        if summary["audited"]:
            line = (f"**H3: {summary['corroborated']} of {summary['audited']} circuits corroborated "
                    f"= {summary['rate']:.0%}, 95% CI [{ci['low']:.1%}, {ci['high']:.1%}]**")
            if summary["reading"]:
                line += f"\n\n{summary['reading']}."
            display(Markdown(line))
        display(Markdown(f"**Verdict: {summary['verdict']}**"))

        save_to_drive(batch_out)
        write_manifest(h3=summary)
        print(f"\nH3 artefacts: {batch_out}")
        if SAVE_RUNS_TO_DRIVE:
            print("On Drive:", DRIVE_BATCH / "h3")
        print("Paste the detection floor and the H3 rate into the write-up.")

except (Exception, SystemExit) as exc:
    traceback.print_exc()
    RESULTS[ARM]['errors']['Trace']=str(exc)
finally:
    root=COMPARISON_ROOT/ARM
    root.mkdir(parents=True,exist_ok=True)
    (root/'stage_status.json').write_text(json.dumps(RESULTS[ARM],indent=2))
    if SAVE_RUNS_TO_DRIVE:
        shutil.copytree(root,DRIVE_ROOT/root.relative_to(LOCAL_REPO),dirs_exist_ok=True)


In [ ]:
# @title Compare the two results
import csv, json, math
import numpy as np
from IPython.display import display, Markdown

rows = []
for arm in CHECKPOINTS:
    root = COMPARISON_ROOT / arm
    def read(relative):
        path = root / relative
        return json.loads(path.read_text()) if path.exists() else {}
    fit = read("fidelity/fidelity.json")
    agg = read("probe/aggregate/aggregate.json")
    h3 = read("probe/h3/h3_rate.json").get("summary", {})
    values = [c["replacement"]["prefix"]["relative_l2"] for c in fit.get("cases", [])]
    values = [v for v in values if v is not None and math.isfinite(v)]
    rows.append(dict(model=arm,
        fidelity_cases=f'{fit.get("completed",0)}/{fit.get("planned","?")}',
        all_layers_action_relative_l2=float(np.mean(values)) if values else None,
        adjusted_selectivity=agg.get("summary",{}).get("pooled",{}).get("sel_adj_l2"),
        h1=agg.get("summary",{}).get("verdict", "not measured"),
        h2=agg.get("summary",{}).get("h2_verdict", "not measured"),
        requested_circuits=TRACE_N_TARGETS,
        audited=h3.get("audited",0), failed=h3.get("failed",0),
        corroborated=h3.get("corroborated",0), rate=h3.get("rate"),
        detection_floor=h3.get("detection_floor"), blind_pools=h3.get("blind_pools",[]),
        stage_errors=RESULTS.get(arm,{}).get("errors",{})))

# Verify common fidelity observations/noise, and compare stored native outputs.
def cases(arm):
    path = COMPARISON_ROOT / arm / "fidelity/fidelity.json"
    report = json.loads(path.read_text()) if path.exists() else {}
    return {(r["task"],r["start"]):r for r in report.get("cases",[])}
a,b = cases("old"),cases("new")
paired = []
for key in sorted(a.keys() & b.keys()):
    left,right = a[key],b[key]
    matched = all(left[k] == right[k] for k in ("batch_sha256","noise_sha256"))
    arrays = [np.load(COMPARISON_ROOT / arm / "fidelity" /
              f"task{key[0]}_init{key[1]}" / "actions.npz") for arm in CHECKPOINTS]
    native_drift = float(np.max(np.abs(arrays[0]["native"]-arrays[1]["native"])))
    for arr in arrays: arr.close()
    x,y = left["replacement"]["prefix"]["relative_l2"],right["replacement"]["prefix"]["relative_l2"]
    paired.append(dict(task=key[0],start=key[1],inputs_match=matched,
        native_max_abs_drift=native_drift,
        new_minus_old_relative_l2=y-x if matched and x is not None and y is not None else None))

report = dict(profile=PROFILE,run_id=RUN_ID,source_sha256=SOURCE_SHA256,
    checkpoint_sha256=CHECKPOINT_HASHES,models=rows,paired_fidelity=paired,
    limitations=["Pilot is exploratory; no automatic superiority claim.",
        "Nomination and parent audit reuse the counterfactual cohort; this is not a held-out test.",
        "Different dictionaries have different feature IDs and may nominate different targets.",
        "Tracing uses each target's top examples in the same discovery pool, not identical target frames.",
        "Circuits share tasks/features; binomial intervals in individual reports assume independence.",
        "Feature selectivity/parent enrichment does not establish causal action features or task success."])
COMPARISON_ROOT.mkdir(parents=True,exist_ok=True)
(COMPARISON_ROOT / "comparison.json").write_text(json.dumps(report,indent=2))
with (COMPARISON_ROOT / "comparison.csv").open("w") as f:
    writer=csv.DictWriter(f,fieldnames=rows[0].keys());writer.writeheader();writer.writerows(rows)
table = "| Model | All-layer action relative L2 ↓ | Adjusted selectivity | Circuits corroborated / audited | Requested |\n|---|---:|---:|---:|---:|\n"
for row in rows:
    table += f'| {row["model"]} | {row["all_layers_action_relative_l2"]} | {row["adjusted_selectivity"]} | {row["corroborated"]}/{row["audited"]} | {row["requested_circuits"]} |\n'
display(Markdown(table))
print("Fidelity input pairs verified:",sum(r["inputs_match"] for r in paired),"/",len(paired))
print("Native prediction drift:",[r["native_max_abs_drift"] for r in paired])
for row in rows:
    print(row["model"], "errors:", row["stage_errors"], "blind pools:",row["blind_pools"])
display(Markdown("Lower replacement error means the surrogate preserves predictions better. "
    "Higher calibrated corroboration with adequate coverage is evidence for better circuits on this audit. "
    "Inspect failures, unequal coverage, input mismatches, and repeat/noise floors before comparing. "
    "Neither result alone proves generalizable reaching features."))
save_to_drive(COMPARISON_ROOT)
print("Comparison:", DRIVE_ROOT / COMPARISON_ROOT.relative_to(LOCAL_REPO))
